# 📊 LlamaIndex — Evaluation সম্পূর্ণ গাইড

## Evaluation কী এবং কেন দরকার?

তুমি একটা RAG system বানিয়েছ। কিন্তু সেটা **কতটা ভালো কাজ করছে** জানো কীভাবে?

```
প্রশ্ন: "What is LlamaIndex?"
Context: "LlamaIndex is a data framework for LLM apps..."
উত্তর: "LlamaIndex is a tool for building chatbots."

🤔 এই উত্তর কি ভালো? Context-এর সাথে মিলছে? সঠিক?
   → Evaluation ছাড়া জানার উপায় নেই!
```

## RAG Evaluation-এর ৩টি মূল প্রশ্ন:
```
1. Faithfulness  → উত্তরটা কি context থেকে এসেছে? (hallucination নেই তো?)
2. Relevancy     → উত্তরটা কি প্রশ্নের সাথে সম্পর্কিত?
3. Correctness   → উত্তরটা কি সত্যিকারের সঠিক? (ground truth-এর সাথে মেলে?)
```

## এই নোটবুকে যা শিখবে:
1. ✅ FaithfulnessEvaluator — Hallucination detect করা
2. ✅ RelevancyEvaluator — উত্তর প্রাসঙ্গিক কিনা
3. ✅ CorrectnessEvaluator — উত্তর সঠিক কিনা (score 1-5)
4. ✅ AnswerRelevancyEvaluator — প্রশ্নের সাথে মিলছে কিনা
5. ✅ Context Relevancy — Retrieved context কতটা ভালো
6. ✅ BatchEvalRunner — অনেক প্রশ্ন একসাথে evaluate
7. ✅ Auto Dataset Generate — প্রশ্ন-উত্তর pair তৈরি করা
8. ✅ Retrieval Evaluation — Retriever কতটা ভালো

---
## ⚙️ Step 0: Setup — Index ও Query Engine তৈরি

> **⚠️ গুরুত্বপূর্ণ:** Evaluation-এর জন্য real LLM লাগবে (MockLLM কাজ করবে না!)  
> `.env` ফাইলে `OPENAI_API_KEY` সেট করো।  
> **API key না থাকলে:** Step 1-4 skip করে Step 5 (Retrieval Eval) থেকে শুরু করো।

In [ ]:
from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

# ── .env থেকে API Key লোড ──
load_dotenv()

# ── Embedding Model (HuggingFace — ফ্রি) ──
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ── LLM (Evaluation-এর জন্য real LLM লাগবে) ──
Settings.llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0   # deterministic উত্তরের জন্য
)

print("✅ Settings সম্পন্ন!")

In [ ]:
# ── Documents লোড ও Index তৈরি ──
documents = SimpleDirectoryReader(input_dir="Data").load_data()

splitter = SentenceSplitter(chunk_size=256, chunk_overlap=50)

index = VectorStoreIndex.from_documents(
    documents=documents,
    transformations=[splitter]
)

# Query Engine তৈরি
query_engine = index.as_query_engine(
    similarity_top_k=2
)

print(f"✅ Index তৈরি! {len(documents)}টি document লোড হয়েছে।")
print("🔍 Query Engine প্রস্তুত।")

---
## 🧠 Evaluation-এর ধারণাটা বোঝো আগে

প্রতিটা Evaluation-এ তিনটা জিনিস থাকে:
```
┌─────────────────────────────────────────────┐
│  query    = ব্যবহারকারীর প্রশ্ন              │
│  contexts = Retrieved chunks (source nodes)  │
│  response = LLM-এর উত্তর                    │
└─────────────────────────────────────────────┘

Evaluator এই তিনটা নিয়ে বিচার করে:
  → Faithfulness:  contexts + response  → hallucinate করেছে?
  → Relevancy:     query + contexts + response → সম্পর্কিত?
  → Correctness:   query + response + reference → সঠিক স্কোর?
```

---
## 📌 Step 1: FaithfulnessEvaluator — Hallucination ধরা

**Faithfulness = উত্তরটা কি context-এর মধ্যে থেকে এসেছে?**

```
Context: "LlamaIndex is a Python framework."
উত্তর: "LlamaIndex is a JavaScript framework." → ❌ Hallucination (passing=False)
উত্তর: "LlamaIndex is a Python framework."     → ✅ Faithful (passing=True)
```

In [ ]:
from llama_index.core.evaluation import FaithfulnessEvaluator

# ── Faithfulness Evaluator তৈরি ──
faithfulness_evaluator = FaithfulnessEvaluator(
    llm=Settings.llm   # LLM দিয়ে evaluate করবে
)

# ── Query চালানো ──
query   = "What is LlamaIndex?"
response = query_engine.query(query)

print(f"❓ প্রশ্ন: {query}")
print(f"🤖 উত্তর: {response.response}")
print()

# ── Evaluate করা ──
eval_result = faithfulness_evaluator.evaluate_response(
    response=response
)

# ── ফলাফল দেখা ──
print("━"*55)
print("📊 Faithfulness Evaluation ফলাফল:")
print(f"   ✅ Passing (হ্যাঁ/না): {eval_result.passing}")
print(f"   📝 Score: {eval_result.score}")
print(f"   💬 Feedback: {eval_result.feedback}")

In [ ]:
# ── Manual Evaluation (Response object ছাড়া) ──
# কখনো কখনো manually context ও response দিতে হয়

eval_result2 = faithfulness_evaluator.evaluate(
    query="What is LlamaIndex?",
    contexts=[
        "LlamaIndex is a data framework for building LLM applications.",
        "It helps with Retrieval Augmented Generation (RAG)."
    ],
    response="LlamaIndex is a tool for playing games.",  # ← ইচ্ছে করে ভুল উত্তর
)

print("📊 Manual Faithfulness Evaluation:")
print(f"   Passing: {eval_result2.passing}   ← False হওয়া উচিত!")
print(f"   Feedback: {eval_result2.feedback}")

---
## 📌 Step 2: RelevancyEvaluator — উত্তর প্রাসঙ্গিক কিনা

**Relevancy = উত্তরটা কি প্রশ্নের সাথে সম্পর্কিত এবং context কি helpful ছিল?**

```
প্রশ্ন: "What is LlamaIndex?"
Context: "Python is a programming language." → ❌ Irrelevant context
উত্তর: "LlamaIndex helps build RAG systems." 
→ উত্তর ঠিক হলেও context irrelevant → passing=False
```

In [ ]:
from llama_index.core.evaluation import RelevancyEvaluator

# ── Relevancy Evaluator ──
relevancy_evaluator = RelevancyEvaluator(
    llm=Settings.llm
)

# ── Query ও Evaluate ──
query    = "What are the key components of LlamaIndex?"
response = query_engine.query(query)

eval_result = relevancy_evaluator.evaluate_response(
    query=query,
    response=response
)

print(f"❓ প্রশ্ন: {query}")
print(f"🤖 উত্তর: {response.response[:150]}...")
print()
print("📊 Relevancy Evaluation ফলাফল:")
print(f"   ✅ Passing: {eval_result.passing}")
print(f"   📝 Score: {eval_result.score}")
print(f"   💬 Feedback: {eval_result.feedback}")

---
## 📌 Step 3: CorrectnessEvaluator — Score 1 থেকে 5

**Correctness = একটা reference (সঠিক উত্তর) এর সাথে তুলনা করে score দেওয়া।**

```
Score 1 → সম্পূর্ণ ভুল
Score 2 → বেশিরভাগ ভুল
Score 3 → আংশিক সঠিক
Score 4 → বেশিরভাগ সঠিক
Score 5 → সম্পূর্ণ সঠিক
```

> তোমকে একটা **reference answer** দিতে হবে (ground truth)।

In [ ]:
from llama_index.core.evaluation import CorrectnessEvaluator

# ── Correctness Evaluator ──
correctness_evaluator = CorrectnessEvaluator(
    llm=Settings.llm
)

# ── Evaluate করা ──
query     = "What is LlamaIndex?"
response  = query_engine.query(query)

# Reference = তোমার জানা সঠিক উত্তর
reference = "LlamaIndex is a data framework designed to help developers \
             build applications with large language models (LLMs) by \
             connecting them to external data sources."

eval_result = correctness_evaluator.evaluate_response(
    query=query,
    response=response,
    reference=reference   # ← সঠিক উত্তর
)

print(f"❓ প্রশ্ন: {query}")
print(f"🤖 LLM উত্তর: {response.response[:100]}...")
print(f"✓  Reference: {reference[:100]}...")
print()
print("📊 Correctness Evaluation ফলাফল:")
print(f"   🌟 Score (1-5): {eval_result.score}")
print(f"   ✅ Passing (≥3): {eval_result.passing}")
print(f"   💬 Feedback: {eval_result.feedback}")

---
## 📌 Step 4: AnswerRelevancyEvaluator

**Answer Relevancy = উত্তরটা কি প্রশ্নটার সরাসরি জবাব দিচ্ছে?**

```
প্রশ্ন:  "Who created LlamaIndex?"
উত্তর:  "LlamaIndex is a great tool for RAG."  
→ উত্তরটা প্রশ্নের জবাব দেয়নি! → Low score

উত্তর:  "LlamaIndex was created by Jerry Liu."
→ সরাসরি জবাব! → High score
```

In [ ]:
from llama_index.core.evaluation import AnswerRelevancyEvaluator

# ── Answer Relevancy Evaluator ──
answer_relevancy_eval = AnswerRelevancyEvaluator(
    llm=Settings.llm
)

query    = "What is RAG?"
response = query_engine.query(query)

eval_result = answer_relevancy_eval.evaluate_response(
    query=query,
    response=response
)

print(f"❓ প্রশ্ন: {query}")
print(f"🤖 উত্তর: {response.response[:120]}...")
print()
print("📊 Answer Relevancy ফলাফল:")
print(f"   🌟 Score: {eval_result.score}")
print(f"   ✅ Passing: {eval_result.passing}")
print(f"   💬 Feedback: {eval_result.feedback}")

---
## 📌 Step 5: Context Relevancy — Retrieved Context কতটা ভালো

**Context Relevancy = প্রশ্নের জন্য সঠিক chunks আনা হয়েছে কিনা?**

এটা Retriever-এর quality measure করে।
```
প্রশ্ন: "What is RAG?"
Retrieved: ["Python syntax tutorial", "JavaScript basics"]
→ Context সম্পূর্ণ irrelevant! Retriever ভালো কাজ করেনি।
```

In [ ]:
from llama_index.core.evaluation import ContextRelevancyEvaluator

# ── Context Relevancy Evaluator ──
ctx_relevancy_eval = ContextRelevancyEvaluator(
    llm=Settings.llm
)

query    = "What topics are covered in the documents?"
response = query_engine.query(query)

# Retrieved contexts দেখা
print("📂 Retrieved Contexts:")
for i, node in enumerate(response.source_nodes):
    print(f"   Node {i+1} [{node.metadata.get('file_name','?')}]: {node.text[:80]}...")

print()

# Evaluate
eval_result = ctx_relevancy_eval.evaluate_response(
    query=query,
    response=response
)

print("📊 Context Relevancy ফলাফল:")
print(f"   🌟 Score: {eval_result.score}")
print(f"   ✅ Passing: {eval_result.passing}")
print(f"   💬 Feedback: {eval_result.feedback}")

---
## 📌 Step 6: BatchEvalRunner — অনেক প্রশ্ন একসাথে Evaluate

Real project-এ একটা দুটো প্রশ্ন না, অনেক প্রশ্ন একসাথে evaluate করতে হয়।

In [ ]:
from llama_index.core.evaluation import (
    BatchEvalRunner,
    FaithfulnessEvaluator,
    RelevancyEvaluator,
)

# ── Evaluators ──
faithfulness_eval = FaithfulnessEvaluator(llm=Settings.llm)
relevancy_eval    = RelevancyEvaluator(llm=Settings.llm)

# ── Test প্রশ্নের তালিকা ──
test_questions = [
    "What is LlamaIndex?",
    "What is RAG?",
    "What are the key components?",
    "What is data ingestion?",
    "What is vector indexing?",
]

print(f"📋 {len(test_questions)}টি প্রশ্ন evaluate করা হবে...\n")

# ── BatchEvalRunner তৈরি ──
runner = BatchEvalRunner(
    evaluators={
        "faithfulness": faithfulness_eval,
        "relevancy":    relevancy_eval,
    },
    workers=2   # parallel evaluation
)

# ── Batch Evaluate ──
# এটা সব প্রশ্নের জন্য query করবে এবং evaluate করবে
eval_results = await runner.aevaluate_queries(
    query_engine,
    queries=test_questions
)

print("✅ Batch Evaluation সম্পন্ন!\n")

In [ ]:
import pandas as pd

# ── ফলাফল সুন্দরভাবে দেখা ──
rows = []

for metric_name, results_list in eval_results.items():
    for i, result in enumerate(results_list):
        rows.append({
            "প্রশ্ন":      test_questions[i][:40] + "...",
            "Metric":     metric_name,
            "Score":      result.score,
            "Passing":    "✅" if result.passing else "❌",
            "Feedback":   result.feedback[:60] + "..." if result.feedback else "-",
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

# ── Summary Statistics ──
print("\n" + "─"*55)
print("📊 Passing Rate Summary:")
for metric_name, results_list in eval_results.items():
    passing = sum(1 for r in results_list if r.passing)
    total   = len(results_list)
    rate    = passing / total * 100
    bar     = "█" * int(rate / 10) + "░" * (10 - int(rate / 10))
    print(f"   {metric_name:15} [{bar}] {rate:.0f}% ({passing}/{total})")

---
## 📌 Step 7: Dataset Auto-Generate করা

হাতে প্রশ্ন না বানিয়ে — LLM দিয়ে **automatically প্রশ্ন তৈরি করানো**!

```
Document → LLM → auto-generate করে:
    - questions (প্রশ্ন)
    - contexts  (কোন chunk থেকে প্রশ্ন)
```

In [ ]:
from llama_index.core.evaluation import generate_question_context_pairs
from llama_index.core.node_parser import SentenceSplitter

# ── Nodes তৈরি করা ──
splitter = SentenceSplitter(chunk_size=256, chunk_overlap=50)
nodes    = splitter.get_nodes_from_documents(documents)

print(f"📦 {len(nodes)}টি node থেকে প্রশ্ন generate হবে...")
print("⏳ একটু সময় লাগবে (LLM call হচ্ছে)...\n")

# ── Auto Dataset Generate ──
qa_dataset = generate_question_context_pairs(
    nodes         = nodes,
    llm           = Settings.llm,
    num_questions_per_chunk = 2   # প্রতিটি chunk থেকে ২টি প্রশ্ন
)

print(f"✅ Dataset তৈরি! মোট {len(qa_dataset.queries)} প্রশ্ন-উত্তর pair।")

In [ ]:
# ── Generated Dataset দেখা ──
print("📋 Generated Dataset (প্রথম ৩টি):")
print()

query_ids     = list(qa_dataset.queries.keys())
relevant_docs = qa_dataset.relevant_docs

for i, qid in enumerate(query_ids[:3]):
    question = qa_dataset.queries[qid]
    # সংশ্লিষ্ট node-এর text দেখা
    relevant_node_ids = relevant_docs[qid]
    
    print(f"{'━'*55}")
    print(f"❓ প্রশ্ন {i+1}: {question}")
    print(f"   Related Node IDs: {relevant_node_ids[:1]}...")

print(f"\n💾 মোট প্রশ্ন: {len(qa_dataset.queries)}")
print(f"   এই dataset দিয়ে systematic evaluation করা যাবে!")

In [ ]:
import json

# ── Dataset Save ও Load করা ──

# Save (JSON ফরম্যাটে)
qa_dataset.save_json("qa_dataset.json")
print("💾 Dataset 'qa_dataset.json' ফাইলে save হয়েছে!")

# Load
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset
loaded_dataset = EmbeddingQAFinetuneDataset.from_json("qa_dataset.json")
print(f"✅ Dataset load হয়েছে! প্রশ্ন: {len(loaded_dataset.queries)}")

---
## 📌 Step 8: Retrieval Evaluation — Retriever কতটা ভালো?

Query Engine এর দুটো অংশ:
```
1. Retriever   → সঠিক chunk আনছে কিনা  ← এখানে evaluate করব
2. LLM         → ভালো উত্তর দিচ্ছে কিনা
```

**Metrics:**
- **Hit Rate** → প্রশ্নের সঠিক chunk কি retrieved list-এ আছে?
- **MRR** (Mean Reciprocal Rank) → সঠিক chunk কত উপরে আছে?

In [ ]:
from llama_index.core.evaluation import RetrieverEvaluator

# ── Retriever তৈরি ──
retriever = index.as_retriever(similarity_top_k=3)

# ── Retriever Evaluator ──
retriever_evaluator = RetrieverEvaluator.from_metric_names(
    metric_names=["hit_rate", "mrr"],
    retriever=retriever
)

print("✅ Retriever Evaluator তৈরি!")
print("   Metrics: Hit Rate + MRR (Mean Reciprocal Rank)")

In [ ]:
# ── Generated dataset দিয়ে Retrieval Evaluate ──
print("⏳ Retrieval Evaluation চলছে...")

eval_results = await retriever_evaluator.aevaluate_dataset(
    qa_dataset
)

# ── ফলাফল দেখা ──
print(f"\n✅ সম্পন্ন! {len(eval_results)}টি query evaluate হয়েছে.")
print(f"\n{'━'*55}")
print("📊 Retrieval Evaluation Result (প্রথম ৫টি):")
for i, result in enumerate(eval_results[:5]):
    print(f"\n   ❓ Query: {result.query[:60]}...")
    for metric, score in result.metric_vals_dict.items():
        icon = "✅" if score > 0 else "❌"
        print(f"      {icon} {metric}: {score:.2f}")

# ── Overall Score ──
print(f"\n{'━'*55}")
print("🏆 Overall Retrieval Performance:")
metrics = eval_results[0].metric_vals_dict.keys()
for metric in metrics:
    scores = [r.metric_vals_dict[metric] for r in eval_results]
    avg    = sum(scores) / len(scores)
    bar    = "█" * int(avg * 10) + "░" * (10 - int(avg * 10))
    print(f"   {metric:15} [{bar}] {avg:.2%}")

---
## 📌 Step 9: সম্পূর্ণ RAG Evaluation Dashboard
সব metrics একসাথে দেখা এবং final report তৈরি করা।

In [ ]:
from llama_index.core.evaluation import (
    BatchEvalRunner,
    FaithfulnessEvaluator,
    RelevancyEvaluator,
    CorrectnessEvaluator,
)

# ── Test প্রশ্ন ও সঠিক উত্তর ──
eval_data = [
    {
        "question":  "What is LlamaIndex?",
        "reference": "LlamaIndex is a data framework for building LLM applications that connect to external data."
    },
    {
        "question":  "What is RAG?",
        "reference": "RAG (Retrieval Augmented Generation) is a technique that retrieves relevant context before generating answers."
    },
    {
        "question":  "What is data ingestion?",
        "reference": "Data ingestion is the process of loading, splitting, and indexing documents into a vector store."
    },
]

# ── সব evaluate করা ──
print("⏳ Full RAG Evaluation চলছে...\n")

faithfulness_eval  = FaithfulnessEvaluator(llm=Settings.llm)
relevancy_eval     = RelevancyEvaluator(llm=Settings.llm)
correctness_eval   = CorrectnessEvaluator(llm=Settings.llm)

all_results = []

for item in eval_data:
    question  = item["question"]
    reference = item["reference"]
    
    # Query করা
    response = query_engine.query(question)
    
    # Evaluate করা
    faith_res  = faithfulness_eval.evaluate_response(response=response)
    relev_res  = relevancy_eval.evaluate_response(query=question, response=response)
    correct_res = correctness_eval.evaluate_response(
        query=question, response=response, reference=reference
    )
    
    all_results.append({
        "Question":     question[:45] + "...",
        "Faithfulness": f"{'✅' if faith_res.passing else '❌'} ({faith_res.score or 'N/A'})",
        "Relevancy":    f"{'✅' if relev_res.passing else '❌'} ({relev_res.score or 'N/A'})",
        "Correctness":  f"{'✅' if correct_res.passing else '❌'} ({correct_res.score:.1f}/5.0)",
    })
    print(f"   ✓ '{question[:40]}' evaluated")

print()
print("═"*75)
print(" 🏆 FULL RAG EVALUATION DASHBOARD")
print("═"*75)

df_final = pd.DataFrame(all_results)
print(df_final.to_string(index=False))
print("═"*75)

---
## 🎯 সারসংক্ষেপ — সব Evaluators এর তুলনা

| Evaluator | কী measure করে | Input | Output |
|-----------|----------------|-------|--------|
| **FaithfulnessEvaluator** | Hallucination আছে কিনা | context + response | True/False |
| **RelevancyEvaluator** | Context + Answer প্রাসঙ্গিক কিনা | query + context + response | True/False |
| **CorrectnessEvaluator** | কতটা সঠিক | query + response + reference | Score 1-5 |
| **AnswerRelevancyEvaluator** | প্রশ্নের সরাসরি জবাব কিনা | query + response | Score |
| **ContextRelevancyEvaluator** | Retrieved context ভালো কিনা | query + context | True/False |
| **RetrieverEvaluator** | Retriever performance | qa_dataset | Hit Rate, MRR |
| **BatchEvalRunner** | অনেক প্রশ্ন একসাথে | evaluators + queries | সব results |

## কখন কোনটা ব্যবহার করব?

```
"আমার RAG ভুল তথ্য দিচ্ছে মনে হচ্ছে"  → FaithfulnessEvaluator
"Retriever সঠিক chunk আনছে না"        → RetrieverEvaluator (Hit Rate, MRR)
"উত্তর প্রশ্নের সাথে মিলছে না"         → RelevancyEvaluator
"A/B test করতে চাই দুটো RAG system"   → CorrectnessEvaluator + BatchEvalRunner
"Regular monitoring করতে চাই"         → BatchEvalRunner + DataFrame report
```

## পরবর্তী শেখার বিষয়
- 📌 **Agents** — Tool ব্যবহার করে AI Agent বানানো
- 📌 **Advanced Retrieval** — BM25, Reranking, Hybrid Search
- 📌 **Router Query Engine** — Multiple index থেকে সঠিক জায়গায় route করা